# Assignment 1 — QANet

**COMP5329 / Deep Learning — University of Sydney, Semester 1 2026**

Run each section in order. Sections 0–1 are one-time setup steps; Sections 2–4 are the main training and evaluation pipeline.

In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

# Adjust this path if your repo is stored elsewhere in Drive.
PROJECT_ROOT = "C:\\Users\\Administrator\\Documents\\coding\\comp5329\\ass1"

In [3]:
# Install Python dependencies (run once per session)
!pip install -r {PROJECT_ROOT}/requirements.txt -q
!python -m spacy download en

⚠ As of spaCy v3.0, shortcuts like 'en' are deprecated. Please use the
full pipeline package name 'en_core_web_sm' instead.
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ------- -------------------------------- 2.4/12.8 MB 11.2 MB/s eta 0:00:01
     --------------- ------------------------ 5.0/12.8 MB 12.1 MB/s eta 0:00:01
     ---------------------- ----------------- 7.3/12.8 MB 11.9 MB/s eta 0:00:01
     ------------------------------ -------- 10.0/12.8 MB 11.9 MB/s eta 0:00:01
     --------------------------------------  12.6/12.8 MB 12.1 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 12.2 MB/s  0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


---
## Section 0 — Environment Setup

Mount Google Drive and install dependencies.

In [4]:
import sys, os

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
print("Working directory:", os.getcwd())

Working directory: C:\Users\Administrator\Documents\coding\comp5329\ass1


---
## Section 1 — Download Data *(delete before submitting)*

Downloads the pre-built mini dataset (sampled SQuAD v1.1 train + full dev set,
with GloVe vectors filtered to the mini vocabulary) from GitHub Releases into `_data/`.

> **One-time step.** Once `_data/` exists on your Drive, delete this section before submission.

In [1]:
from Tools.download import download_mini

download_mini(data_dir="_data")

Step 1 / 2  —  Mini dataset (SQuAD + GloVe)
  [skip] Mini dataset already present in _data/.

Step 2 / 2  —  spaCy language model
⚠ As of spaCy v3.0, shortcuts like 'en' are deprecated. Please use the
full pipeline package name 'en_core_web_sm' instead.
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---- ----------------------------------- 1.6/12.8 MB 9.3 MB/s eta 0:00:02
     ------------ --------------------------- 3.9/12.8 MB 10.2 MB/s eta 0:00:01
     -------------------- ------------------- 6.6/12.8 MB 11.2 MB/s eta 0:00:01
     ----------------------------- ---------- 9.4/12.8 MB 11.7 MB/s eta 0:00:01
     --------------------------------------  12.6/12.8 MB 12.5 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 12.3 MB/s  0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')

Mini dataset download complete.


---
## Section 2 — Preprocess Data *(delete before submitting)*

Tokenises the SQuAD JSON files, builds word/char vocabularies from GloVe, and writes padded index tensors to `_data/`.

> **One-time step.** Once `_data/*.npz` exists on your Drive, delete this section before submission. Re-run only if you change `para_limit`, `ques_limit`, or other shape parameters.

In [2]:
from Tools.preproc import preprocess

preprocess(
    train_file="_data/squad/train-mini.json",
    dev_file="_data/squad/dev-v1.1.json",
    glove_word_file="_data/glove/glove.mini.txt",
    target_dir="_data",
    para_limit=400,
    ques_limit=50,
)

Generating train examples…


100%|██████████| 150/150 [00:02<00:00, 62.86it/s]


  30293 questions in total
Generating dev examples…


100%|██████████| 48/48 [00:01<00:00, 37.07it/s]


  10570 questions in total
Generating word embedding…


...

100%|██████████| 30293/30293 [00:03<00:00, 10021.68it/s]


  Built 30169 / 30293 instances
Processing dev examples…


100%|██████████| 10570/10570 [00:01<00:00, 8435.10it/s]


  Built 10465 / 10570 instances
Saving word embedding…
Saving char embedding…
Saving train eval…
Saving dev eval…
Saving word dictionary…
Saving char dictionary…
Saving dev meta…

Preprocessing complete.
  Outputs → _data/


{'train_record_file': '_data\\train.npz',
 'dev_record_file': '_data\\dev.npz',
 'word_emb_file': '_data\\word_emb.json',
 'char_emb_file': '_data\\char_emb.json',
 'train_eval_file': '_data\\train_eval.json',
 'dev_eval_file': '_data\\dev_eval.json',
 'word2idx_file': '_data\\word2idx.json',
 'char2idx_file': '_data\\char2idx.json',
 'dev_meta_file': '_data\\dev_meta.json'}

---
## Section 3 — Train

Trains QANet on SQuAD v1.1 and saves the best checkpoint to `_model/model.pt`.

In [1]:
from TrainTools.train import train

results = train(
    # ── data paths (must match preprocess outputs) ──────────────────────
    train_npz       = "_data/train.npz",
    dev_npz         = "_data/dev.npz",
    word_emb_json   = "_data/word_emb.json",
    char_emb_json   = "_data/char_emb.json",
    train_eval_json = "_data/train_eval.json",
    dev_eval_json   = "_data/dev_eval.json",
    save_dir        = "_model",
    log_dir         = "_log",

    # ── training loop ────────────────────────────────────────────────────
    #num_steps  = 60000,
    num_steps = 60000,
    batch_size = 8,
    seed       = 42,

    # ── vanilla recipe: SGD, no scheduler, NLL loss ───────────────────────
    optimizer_name = "adam",
    scheduler_name = "lambda",
    loss_name      = "qa_nll",
    learning_rate=1e-3,
    
    checkpoint=1000,
)

print(f"Best F1: {results['best_f1']:.4f}  |  Best EM: {results['best_em']:.4f}")

100%|██████████| 1000/1000 [03:18<00:00,  5.04it/s]


STEP     1000  loss 92.668143



100%|██████████| 150/150 [00:06<00:00, 21.47it/s]


VALID(train) loss 4.702253  F1 7.678636  EM 0.416667



100%|██████████| 150/150 [00:06<00:00, 21.52it/s]


...

STEP    11000  loss 3.315740



100%|██████████| 150/150 [00:07<00:00, 20.17it/s]


VALID(train) loss 2.700777  F1 23.010724  EM 15.416667



 65%|██████▌   | 98/150 [00:05<00:02, 18.63it/s]


KeyboardInterrupt: 

In [ ]:
print(f"Best F1: {results['best_f1']:.4f}  |  Best EM: {results['best_em']:.4f}")

Best F1: 21.2314  |  Best EM: 14.9167


---
## Section 4 — Evaluate

Loads the saved checkpoint and runs inference on the full dev set.

In [ ]:
from EvaluateTools.evaluate import evaluate

metrics = evaluate(
    dev_npz       = "_data/dev.npz",
    word_emb_json = "_data/word_emb.json",
    char_emb_json = "_data/char_emb.json",
    dev_eval_json = "_data/dev_eval.json",
    save_dir      = "_model",
    log_dir       = "_log",
    ckpt_name     = "model.pt",
)

print(f"F1: {metrics['f1']:.4f}  |  EM: {metrics['exact_match']:.4f}  |  Loss: {metrics['loss']:.6f}")

100%|██████████| 1309/1309 [01:01<00:00, 21.39it/s]


TEST  loss 4.204651  F1 19.062828  EM 10.625896
F1: 19.0628  |  EM: 10.6259  |  Loss: 4.204651
